In [7]:
!pip install --upgrade pip setuptools wheel
!pip install scikit-learn==1.4.2 --only-binary=:all:

In [8]:
import boto3
import sagemaker

In [9]:
#check role

role = sagemaker.get_execution_role()

print(role)

arn:aws:iam::487789105604:role/LabRole


In [10]:
#check your bucket

s3 = boto3.client("s3")

response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

sagemaker-us-east-1-487789105604


In [11]:
import tarfile
import os

# Tentukan path file model Anda
model_path = "model/best_credit_score_pipeline.joblib"
archive_name = "model.tar.gz"

print("Memulai proses packaging untuk AWS SageMaker...")

if os.path.exists(model_path):
    # Package the credit score model for AWS SageMaker deployment
    with tarfile.open(archive_name, "w:gz") as tar:
        tar.add(
            model_path,                                 # Path lokal di notebook Anda
            arcname="best_credit_score_pipeline.joblib" # Nama file setelah diekstrak di dalam AWS Container nanti
        )
    print(f"Model berhasil dibungkus menjadi {archive_name}!")
    print("-> File ini sekarang siap di-upload ke Amazon S3.")
else:
    print(f"Error: File {model_path} tidak ditemukan!")
    print("-> Pastikan Anda sudah menjalankan file train.py terlebih dahulu.")

Memulai proses packaging untuk AWS SageMaker...
Model berhasil dibungkus menjadi model.tar.gz!
-> File ini sekarang siap di-upload ke Amazon S3.


In [12]:
#Upload model to S3, because deployment will take the model form s3

s3 = boto3.client("s3")

bucket_name = "sagemaker-us-east-1-487789105604"

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model/model.tar.gz"
)

In [13]:
# check model can be run or not

from inference import model_fn

model = model_fn("model")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.2 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from ver

In [16]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
import json

# ---- EDIT THESE ---------------------------------------------------------
BUCKET = "sagemaker-us-east-1-487789105604" 
MODEL_S3_KEY = "model/model.tar.gz" 
ENDPOINT_NAME = "credit-score-endpoint"
# -------------------------------------------------------------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"

def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]

def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",  
        source_dir=".",              
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (Membutuhkan waktu sekitar 5-8 menit)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )
    
    print("Testing endpoint with sample data...")
    # 32 Features matching your inference.py FEATURE_NAMES list:
    sample_data = [
        "January", 30, "Engineer", 50000.0, 4166.67, 3, 2, 15.0, 2, 14, 
        4, 10.0, 2, "Standard", 1500.0, 30.0, "NM", 200.0, 100.0, 
        "Low_spent_Small_value_payments", 500.0, 150.0, 
        # Loan Types (10 dummy columns: Auto, Builder, Consolidation, dll)
        1, 0, 1, 0, 0, 0, 0, 0, 0, 0
    ]
    
    sample = {
        "instances": [sample_data]
    }
    
    runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample),
    )
    
    print("\nSmoke test response:")
    # Seharusnya mencetak hasil prediksi seperti: {"probabilities": [...], "predictions": [1], "labels": ["Standard"]}
    print(response["Body"].read().decode("utf-8"))

    print(
        f"\nEndpoint '{ENDPOINT_NAME}' is live in {REGION}.\n"
        f"Aplikasi Streamlit Anda sekarang sudah bisa terhubung!\n"
        f"Jangan lupa matikan endpoint ini sebelum lab selesai: predictor.delete_endpoint()"
    )

if __name__ == "__main__":
    main()

Role:      arn:aws:iam::487789105604:role/LabRole
Model URI: s3://sagemaker-us-east-1-487789105604/model/model.tar.gz
Endpoint:  credit-score-endpoint

Deploying endpoint (Membutuhkan waktu sekitar 5-8 menit)...
------!Testing endpoint with sample data...

Smoke test response:
{"probabilities": [[0.3209718614718614, 0.4481471861471861, 0.23088095238095238]], "predictions": [1], "labels": ["Standard"]}

Endpoint 'credit-score-endpoint' is live in us-east-1.
Aplikasi Streamlit Anda sekarang sudah bisa terhubung!
Jangan lupa matikan endpoint ini sebelum lab selesai: predictor.delete_endpoint()


In [15]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
# Updated to match the endpoint name in your deployment script
ENDPOINT_NAME = "credit-score-endpoint"

# 1. Delete the stuck or active endpoint
print(f"Deleting endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")

Deleting endpoint: credit-score-endpoint...
Endpoint deletion triggered.
Deleting endpoint configuration: credit-score-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete! You can now safely run your main deploy script.
